In [ ]:
%reset -f
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget
import os
import sys
import h5py

from scripts import generate_dataset
from src.sampling import latin_hypercube_sampling

## **This cell generates a dataset with parameters:**
### Parameters
* <b>N</b>: number of simulations
* <b>d</b>: number of parameters being randomly assigned
* <b>output_path</b>: directory in which dataset.h5 will be created
* <b>lua_file_path</b>: path of lua file
* <b>l_bounds</b>: array of lower bounds for a parameter range in which each index represents a different parameter (ten_dq_i, ten_dq_f, etc.)
* <b>u_bounds</b>: array of upper bounds for a parameter range in which each index represents a different parameter (ten_dq_i, ten_dq_f, etc.)

In [ ]:
L_BOUNDS = np.array([0.5, 0.75])
U_BOUNDS = np.array([5, 1])

output_path = '/Users/pierolujanpedreschi/SLAC-Project/Co/CoTerpy/6.30.2026_Test/TM_test1'
lua_file = 'TM_Ledge_spec_job.lua'
lua_file_path = '/Users/pierolujanpedreschi/SLAC-Project/QuantyRIXS_ML/'
generate_dataset(N=10, d=2, output_path=output_path, lua_file_path=lua_file_path, l_bounds=L_BOUNDS, u_bounds=U_BOUNDS)

WindowsPath('C:/Users/bpoult/Documents/Research/X-ray_Simulation_Software/Simulations/Tests/6.17.2026_Tests/GS_Oh.inp_rixs')

## **This cell runs the quanty script with the above parameters DELETE NEXT TIME**

In [ ]:
# lua_file = 'green_crystal_field_Fe3_Medge.lua'
# lua_file = 'greenMLCT_Co3d6_D4h_RCN_conf_job.lua'  # original Co3+ L-edge script (preserved for posterity)
lua_file = 'TM_Ledge_spec_job.lua'
# lua_file = 'groundstate.lua'
# lua_file = 'rixs_partial_excitations.lua'


lua_file_path = 'C:/Users/bpoult/Documents/Research/X-ray_Simulation_Software/QuantyRIXS_ML/'
result = run_quanty_sim(folder_path=output_path,lua_file=lua_file,lua_file_path=lua_file_path)
print("STDOUT:", result.stdout)

Copied C:\Users\bpoult\Documents\Research\X-ray_Simulation_Software\QuantyRIXS_ML\TM_Ledge_spec_job.lua to C:\Users\bpoult\Documents\Research\X-ray_Simulation_Software\Simulations\Tests\6.17.2026_Tests\TM_Ledge_spec_job.lua
STDOUT: None


## **Plot Simulations on top of each other**

In [ ]:
with h5py.File("/Users/pierolujanpedreschi/SLAC-Project/Co/CoTerpy/6.30.2026_Test/TM_test1/dataset.h5", "r") as f:
    print("Keys in the file:", list(f.keys()))

    energies = np.array(f['Energies'][:])
    all_spectra = np.array(f['Spectra'][:])

for row_idx, intensities in enumerate(all_spectra):
    plt.plot(energies, intensities, label=f"Spectrum {row_idx+1}", alpha=0.6)

plt.title("Multiple Spectra Comparison")
plt.xlabel("Energy")
plt.ylabel("Intensity")
plt.legend()
plt.show()

In [ ]:
plt.close('all')
# Define positions and labels
lines = [
    (0, 'GS'),
    (1.8, '3MC'),
    (2.15, '5MC'),
    (2.48, '3MC'),
    (2.89, '1MC'),
    (4.22, '1MC')
]

# Define colors for each unique label using colorblind-friendly palette
color_map = {
    'GS': '#000000',    # Black
    '3MC': '#E69F00',   # Orange
    '5MC': '#56B4E9',   # Sky blue
    '1MC': '#009E73'    # Bluish green
}

plt.figure(figsize=(4, 4), dpi=100)

# Track which labels have been added to legend
added_labels = set()

# Plot horizontal lines
for position, label in lines:
    # Only add label if it's the first occurrence (for legend)
    if label not in added_labels:
        plt.axhline(y=position, color=color_map[label], linewidth=2, 
                   label=label, alpha=0.8)
        added_labels.add(label)
    else:
        plt.axhline(y=position, color=color_map[label], linewidth=2, alpha=0.8)

# Labels and styling
plt.ylabel('Energy (eV)', fontsize=12, fontweight='bold')
plt.title('Electronic State Energies', fontsize=14, fontweight='bold', pad=15)

# Legend - bottom right
plt.legend(frameon=True, fancybox=True, shadow=True, fontsize=11, loc='lower right')

# Grid and styling
plt.xlim(0, 1)  # Adjust as needed
plt.tight_layout()
plt.xticks([])
plt.show()

---
## **Charge Transfer (CT) calculations — LMCT + MLCT**
Use `generate_inp_quanty_CT` and `TM_Ledge_CT_spec_job.lua` (in the `CT/` subdirectory).
The `.inp_rixs` file is identical in format to the non-CT version — use `generate_inp_rixs`.

In [ ]:
#### CT (Charge Transfer) ####

output_path = 'C:/Users/bpoult/Documents/Research/X-ray_Simulation_Software/Simulations/Co/CoTerpy/CT/'
fname_quanty = 'CT.inp_quanty'
fname_rixs = 'CT.inp_rixs'

# System identification — same as non-CT workflow.
params_setup = {
    'atom': 'Co',
    'charge': '3+',
    'edge': 'L',
    'initial_state': 1,
    'rcn_file': 'C:/Users/bpoult/Documents/Research/X-ray_Simulation_Software/QuantyRIXS_ML/RCNparameter.txt',
}

params_i = {
    # --- standard parameters (same keys as non-CT) ---
    'NPsi_i': 600,
    'tenDq_3d_i': 3.3,
    'Ds_3d_i': 0.18,
    'Dt_3d_i': 0.06,
    'scalef2_3d3d_i': 0.6,
    'scalef4_3d3d_i': 0.6,
    'scale_3dSOC_i': 1.0,
    'U_3d_3d_i': 5.0,
    # --- CT additions: LMCT (occupied ligand L1) ---
    'tenDq_L1_i': 0.4,
    'Delta_3d_L1_i': 0.7,
    'Veg_3d_L1_i': 3.6,
    'Vt2g_3d_L1_i': 0.9,
    # --- CT additions: MLCT (unoccupied ligand L2) ---
    'tenDq_L2_i': 0.0,
    'Delta_3d_L2_i': 3.2,
    'Veg_3d_L2_i': 0.0,
    'Vt2g_3d_L2_i': 1.2,
}

params_f = {
    # --- standard parameters (same keys as non-CT) ---
    'NPsi_f': 600,
    'tenDq_3d_f': 2.8,
    'Ds_3d_f': 0.165,
    'Dt_3d_f': 0.055,
    'scalef2_3d3d_f': 0.6,
    'scalef4_3d3d_f': 0.6,
    'scale_3dSOC_f': 1.0,
    'U_3d_3d_f': 5.0,
    'U_2p_3d_f': 6.0,
    'scalef2_2p3d': 0.6,
    'scaleg': 0.6,
    'scale_2pSOC': 1.0,
    'E_2p': 712.0,
    # --- CT additions: LMCT (occupied ligand L1) ---
    'tenDq_L1_f': 0.4,
    'Delta_3d_L1_f': -0.3,
    'Veg_3d_L1_f': 3.3,
    'Vt2g_3d_L1_f': 0.8,
    # --- CT additions: MLCT (unoccupied ligand L2) ---
    'tenDq_L2_f': 0.0,
    'Delta_3d_L2_f': 4.2,
    'Veg_3d_L2_f': 0.0,
    'Vt2g_3d_L2_f': 1.0,
}

params_rixs = {
    'energy_start': 705,
    'energy_end': 730,
    'energy_step': 0.1,
    'loss_start': -5,
    'loss_end': 20,
    'loss_step': 0.1,
    'FWHM_lorentz1': 0.35,
    'FWHM_lorentz1b': 0.7,
    'FWHM_lorentz2': 0.0,
    'Eshift': 0.0,
    'L3_L2_split': 717,
    'pol': 0,
}

generate_inp_quanty_CT(params_i, params_f, params_setup, output_path, fname_quanty)
generate_inp_rixs(params_rixs, output_path, fname_rixs)

## **This cell runs the CT quanty script with the above parameters**

In [ ]:
lua_file = 'TM_Ledge_CT_spec_job.lua'

lua_file_path = 'C:/Users/bpoult/Documents/Research/X-ray_Simulation_Software/QuantyRIXS_ML/CT/'
result = run_quanty_sim(folder_path=output_path, lua_file=lua_file, lua_file_path=lua_file_path)
print("STDOUT:", result.stdout)